# golo — comparar Python vs Dart (CryptoVault V2)
Este cuaderno **clona el repo**, ejecuta las pruebas Python y comprueba igualdad con una `pass` que te pide.
El Dart (`vault.dart`) usa el mismo envelope `PRBX`, así que un archivo cifrado en Python se descifra en Dart y viceversa.

In [ ]:
import os, sys
REPO_URL = "https://github.com/elmasbe/golo.git"
DEST = "/tmp/golo"
if not os.path.isdir(DEST):
    !git clone $REPO_URL $DEST
else:
    print("ya clonado", DEST)
sys.path.insert(0, DEST)
%cd $DEST
!ls -la

In [ ]:
!pip -q install cryptography 2>&1 | tail -n 2
!python test_vault.py

In [ ]:
import getpass
from toolsec import Vault, is_envelope

pw = getpass.getpass("pass: ")
v = Vault(pw)
msg = b"mensaje de prueba para igualdad"
enc = v.encrypt(msg)
print("envelope:", enc[:4], "ver:", enc[4], "len:", len(enc))
print("salt:", enc[5:21].hex())
print("nonce:", enc[21:33].hex())
print("tag:", enc[-16:].hex())
dec = v.decrypt(enc)
print("igualdad:", dec == msg)
assert dec == msg, "FALLO igualdad"
# pass incorrecta debe fallar
mal = Vault("pass-equivocada").decrypt(enc)
print("pass incorrecta ->", mal)
assert mal is None
print("OK igualdad comprobada")

## Comparar con Dart
En tu máquina con Dart:
```
dart pub get
echo -n "mensaje de prueba para igualdad" > msg.bin
dart run vault.dart enc "TU_PASS" msg.bin msg.prbx
dart run vault.dart dec "TU_PASS" msg.prbx msg.out
```
Luego sube `msg.prbx` aquí y descífralo con Python (`Vault(pw).decrypt`) — debe dar igualdad. Y al revés: cifra aquí con `cli.py enc` y descifra en Dart.